In [10]:
import json
import time
import os
import gzip
from web3 import Web3
from hexbytes import HexBytes

In [11]:
# === CONFIGURATION ===
INFURA_URL = "https://mainnet.infura.io/v3/3921fc62a7ce4cda98926f47409b3d19"
CONTRACT_ADDRESS = "0xCBCdF9626bC03E24f779434178A73a0B4bad62eD"

# Connect to Ethereum
w3 = Web3(Web3.HTTPProvider(INFURA_URL))

# Get the latest block number
LATEST_BLOCK = w3.eth.block_number
BLOCK_STEP = 5000  # Query chunks to avoid API limits
MAX_BLOCKS = 1_000_000  # Stop after 1M blocks
SAVE_FILE = "processed_blocks.json"  # JSON file to store progress
LOG_FILE = "logs.json.gz"  # Compressed log file if >4GB

In [12]:
# === LOAD PROGRESS FROM FILE ===
def load_progress():
    """Loads saved progress from file, handling empty or corrupt data."""
    if os.path.exists(SAVE_FILE):
        try:
            with open(SAVE_FILE, "r") as f:
                return json.load(f)
        except (json.JSONDecodeError, ValueError):
            print(f"⚠️ Warning: {SAVE_FILE} is corrupt. Resetting progress...")

    return {"last_block": LATEST_BLOCK, "logs": []}  # Default fallback


progress = load_progress()
current_block = progress["last_block"]
all_logs = progress["logs"]

# Define the reverse scanning range
end_block = max(current_block - MAX_BLOCKS, 0)

In [13]:
# === CONVERT DATA TO JSON-COMPATIBLE FORMAT ===
def convert_value(value):
    """Converts Web3's HexBytes & non-serializable data in dictionaries/lists."""
    if isinstance(value, HexBytes):
        return value.hex()  # Convert bytes to "0x..." hex string
    elif isinstance(value, list):
        return [convert_value(item) for item in value]
    elif isinstance(value, dict):
        return {k: convert_value(v) for k, v in value.items()}
    return value


def convert_logs_to_json(logs):
    """Converts Web3 logs to standard JSON-compatible dictionaries."""
    return [convert_value(dict(log)) for log in logs]


# === SAVE PROGRESS SAFELY ===
def save_progress(logs):
    """Appends new logs & saves progress to prevent re-querying."""

    try:
        # Load existing logs (if any)
        existing_logs = []
        if os.path.exists(SAVE_FILE):
            with open(SAVE_FILE, "r") as f:
                existing_logs = json.load(f).get("logs", [])

        # Convert logs to JSON-serializable format and append
        json_logs = convert_logs_to_json(logs)
        existing_logs.extend(json_logs)

        # Save progress
        with open(SAVE_FILE, "w") as f:
            json.dump({"last_block": current_block, "logs": existing_logs}, f, indent=2)

        # Compress if file exceeds 4GB
        if (
            os.path.exists(SAVE_FILE)
            and os.path.getsize(SAVE_FILE) > 4 * 1024 * 1024 * 1024
        ):
            with open(SAVE_FILE, "rb") as f_in, gzip.open(LOG_FILE, "wb") as f_out:
                f_out.writelines(f_in)
            os.remove(SAVE_FILE)  # Delete uncompressed version
            print(f"📦 Compressed log file: {LOG_FILE}")

    except Exception as e:
        print(f"❌ Failed to save progress: {e}")

In [14]:
# === PAGINATED LOG RETRIEVAL ===
while current_block > end_block:
    from_block = max(current_block - BLOCK_STEP, end_block)
    print(f"📡 Fetching logs from block {from_block} to {current_block}...")

    success = False
    retries = 0
    logs = []

    # Handle retries for RPC rate limits
    while not success and retries < 5:
        try:
            logs = w3.eth.get_logs(
                {
                    "fromBlock": from_block,
                    "toBlock": current_block,
                    "address": CONTRACT_ADDRESS,
                }
            )
            success = True  # Stop retrying if successful
        except Exception as e:
            if "429" in str(e):
                wait_time = 5 * (2**retries)  # Exponential backoff
                print(f"⚠️ 429 Too Many Requests! Retrying in {wait_time}s...")
                time.sleep(wait_time)
                retries += 1
            else:
                print(f"❌ Critical Error: {e}")
                break  # Stop loop on non-rate-limit errors

    # Save progress if the request was successful
    if success:
        current_block = from_block - 1  # Move backward
        save_progress(logs)  # Save logs safely

📡 Fetching logs from block 20891507 to 20896507...
📡 Fetching logs from block 20886506 to 20891506...
📡 Fetching logs from block 20881505 to 20886505...
📡 Fetching logs from block 20876504 to 20881504...
📡 Fetching logs from block 20871503 to 20876503...
📡 Fetching logs from block 20866502 to 20871502...
📡 Fetching logs from block 20861501 to 20866501...
📡 Fetching logs from block 20856500 to 20861500...
📡 Fetching logs from block 20851499 to 20856499...
📡 Fetching logs from block 20846498 to 20851498...
📡 Fetching logs from block 20841497 to 20846497...
📡 Fetching logs from block 20836496 to 20841496...
📡 Fetching logs from block 20831495 to 20836495...
📡 Fetching logs from block 20826494 to 20831494...
📡 Fetching logs from block 20821493 to 20826493...
📡 Fetching logs from block 20816492 to 20821492...
📡 Fetching logs from block 20811491 to 20816491...
📡 Fetching logs from block 20806490 to 20811490...
📡 Fetching logs from block 20801489 to 20806489...
📡 Fetching logs from block 2079

In [9]:
# Final completion message
print(f"✅ Finished processing {MAX_BLOCKS} blocks!")

✅ Finished processing 1000000 blocks!
